In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, Tuple, List
from IPython.display import display
import re, os

from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

# config 
LABEL_COL = "Is_Suspicious"
VARIANT_ROOTS: Dict[str, Path] = {
    "baseline": Path("./4_class_balance/baseline_500"),
    "smote":    Path("./4_class_balance/smote_500"),
    "adasyn":   Path("./4_class_balance/adasyn_500"),
    "rose":     Path("./4_class_balance/rose_500"),
}

VARIANT_TRAIN_FILE = {
    "baseline": "train_baseline.csv",
    "smote":    "train_smote.csv",
    "adasyn":   "train_adasyn.csv",
    "rose":     "train_rose.csv",
}
ALGOS = ["GBM", "XGB", "RF"]
TARGET_RECALL = 0.98
RANDOM_STATE = 42


WAVE_FILTER = None  

# helpers

def list_waves(root: Path) -> List[str]:
    if not root.exists():
        return []
    waves = sorted([p.name for p in root.iterdir() if p.is_dir()])
    return waves


def _resolve_train_path(dir_path: Path, variant: str) -> Path:
    """Return the first existing train file path for a variant within dir_path."""
    primary = dir_path / VARIANT_TRAIN_FILE[variant]
    if primary.exists():
        return primary
    fallback = dir_path / "train.csv"
    if fallback.exists():
        return fallback
    return primary  


def load_split(variant: str, wave: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    root = VARIANT_ROOTS[variant]
    d = root / wave
    train_path = _resolve_train_path(d, variant)
    test_path  = d / "test.csv"
    if not train_path.exists() or not test_path.exists():
        raise FileNotFoundError(f"Missing files for {variant}/{wave} → {train_path} / {test_path}")
    tr = pd.read_csv(train_path)
    te = pd.read_csv(test_path)
    if LABEL_COL not in tr.columns or LABEL_COL not in te.columns:
        raise KeyError(f"Column '{LABEL_COL}' not found in train/test for {variant}/{wave}")
    y_tr = tr[LABEL_COL].astype(int).to_numpy()
    y_te = te[LABEL_COL].astype(int).to_numpy()
    X_tr = tr.drop(columns=[LABEL_COL]).to_numpy()
    X_te = te.drop(columns=[LABEL_COL]).to_numpy()
    return X_tr, y_tr, X_te, y_te


def make_model(algo: str, y_tr: np.ndarray):
    if algo == "GBM":
        return GradientBoostingClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE
        )
    elif algo == "XGB" and HAS_XGB:
        pos = max(1, int((y_tr == 1).sum()))
        neg = max(1, int((y_tr == 0).sum()))
        spw = neg / pos
        return XGBClassifier(
            n_estimators=400, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            reg_lambda=1.0, random_state=RANDOM_STATE,
            eval_metric="logloss", n_jobs=-1, tree_method="hist",
            scale_pos_weight=spw
        )
    elif algo == "RF":
        return RandomForestClassifier(
            n_estimators=300, max_depth=None, min_samples_leaf=1,
            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
        )
    else:
        raise RuntimeError("XGBoost not installed. Either install xgboost or remove 'XGB' from ALGOS.")


def threshold_for_recall(y_true: np.ndarray, scores: np.ndarray, target: float):
    order = np.argsort(-scores)
    y_sorted = y_true[order]
    s_sorted = scores[order]
    P = int(y_true.sum())
    if P == 0:
        return 1.0, {"precision": 0.0, "recall": 0.0, "tp": 0, "fp": 0, "tn": int((y_true==0).sum()), "fn": 0}
    tp_cum = np.cumsum(y_sorted)
    fp_cum = np.cumsum(1 - y_sorted)
    recall_cum = tp_cum / (P + 1e-12)
    idx = np.where(recall_cum >= target)[0]
    if len(idx) == 0:
        thr = s_sorted[-1] - 1e-12
        pred = np.ones_like(y_true)
    else:
        k = int(idx[0])
        thr = s_sorted[k]
        pred = (scores >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0,1]).ravel()
    prec = tp / max(1, tp + fp)
    rec  = tp / max(1, P)
    return float(thr), {"precision": float(prec), "recall": float(rec), "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn)}

# main comparison

def run_comparison():
    print("ROOTS:", {k: str(v) for k,v in VARIANT_ROOTS.items()})
    waves = list_waves(VARIANT_ROOTS["baseline"]) or []
    if WAVE_FILTER is not None:
        waves = [w for w in waves if WAVE_FILTER(w)]
    assert waves, "No waves found in baseline. Make sure you generated features/balanced sets first."
    print(f"Detected {len(waves)} wave(s): {waves}")

    rows = []
    thr_col = f"thr@R>={TARGET_RECALL:.2f}"

    for variant in ["baseline","smote","adasyn","rose"]:
        for wave in waves:
            try:
                X_tr, y_tr, X_te, y_te = load_split(variant, wave)
            except (FileNotFoundError, KeyError) as e:
                print(f"[skip] {e}")
                continue

            for algo in ALGOS:
                if algo == "XGB" and not HAS_XGB:
                    print("[warn] xgboost not installed, skipping XGB")
                    continue

                model = make_model(algo, y_tr)

                # sample weights for GBM (RF uses class_weight, XGB uses scale_pos_weight)
                sw = None
                if algo == "GBM":
                    n_pos = max(1, int((y_tr == 1).sum()))
                    n_neg = max(1, int((y_tr == 0).sum()))
                    w_pos = n_neg / (n_pos + n_neg)
                    w_neg = n_pos / (n_pos + n_neg)
                    sw = np.where(y_tr == 1, w_pos, w_neg)

                model.fit(X_tr, y_tr, sample_weight=sw) if sw is not None else model.fit(X_tr, y_tr)

                # probabilities for positive class
                if hasattr(model, "predict_proba"):
                    p = model.predict_proba(X_te)[:, 1]
                else:
                    try:
                        dec = model.decision_function(X_te)
                        p = (dec - dec.min()) / (dec.max() - dec.min() + 1e-12)
                    except Exception:
                        p = model.predict(X_te).astype(float)

                roc = roc_auc_score(y_te, p)
                prc = average_precision_score(y_te, p)
                thr, at = threshold_for_recall(y_te, p, TARGET_RECALL)
                n_test = len(y_te)
                fp_per_1000 = at["fp"] / max(1, n_test) * 1000.0

                rows.append({
                    "wave": wave,
                    "variant": variant,
                    "algo": algo,
                    "roc_auc": roc,
                    "pr_auc": prc,
                    thr_col: thr,
                    "precision@R": at["precision"],
                    "recall@thr": at["recall"],
                    "TP": at["tp"],
                    "FP": at["fp"],
                    "TN": at["tn"],
                    "FN": at["fn"],
                    "FP_per_1000": fp_per_1000,
                    "n_test": n_test,
                })

    res = pd.DataFrame(rows)
    if res.empty:
        print("No rows produced: check file names/paths and LABEL_COL. See ROOTS/Detected waves above.")
        return

    out_dir = Path("./out_compare"); out_dir.mkdir(exist_ok=True, parents=True)
    res.to_csv(out_dir / "results_by_wave.csv", index=False)

    # summary
    def agg(df):
        return pd.Series({
            "waves": len(df),
            "roc_auc_mean": df["roc_auc"].mean(),
            "pr_auc_mean": df["pr_auc"].mean(),
            "precision@R_mean": df["precision@R"].mean(),
            "recall@thr_mean": df["recall@thr"].mean(),
            "FP_per_1000_mean": df["FP_per_1000"].mean(),
        })

    summary = res.groupby(["variant","algo"], as_index=False).apply(agg)
    summary.to_csv(out_dir / "summary.csv", index=False)

    # Precision/Recall
    pr_focus = summary[[
        "variant","algo","precision@R_mean","recall@thr_mean",
        "FP_per_1000_mean","pr_auc_mean","roc_auc_mean","waves"
    ]].sort_values(["precision@R_mean","recall@thr_mean"], ascending=[False, False])
    pr_focus.to_csv(out_dir / "precision_recall_summary.csv", index=False)

    # confusion matrices 
    conf_dir = out_dir / "conf_matrices"; conf_dir.mkdir(exist_ok=True, parents=True)
    def _sanitize(s: str) -> str:
        return str(s).replace("/","_").replace("\\","_").replace(" ","_")
    for _, r in res.iterrows():
        cm = pd.DataFrame([[int(r["TN"]), int(r["FP"])],[int(r["FN"]), int(r["TP"])]] ,
                          columns=["Pred=0","Pred=1"], index=["True=0","True=1"])
        fname = f"cm_{_sanitize(r['variant'])}_{_sanitize(r['algo'])}_{_sanitize(r['wave'])}.csv"
        cm.to_csv(conf_dir / fname)

    # progress tables
    progress_dir = out_dir / "progress_by_wave"; progress_dir.mkdir(exist_ok=True, parents=True)
    cols = ["wave","precision@R","recall@thr","FP_per_1000","TP","FP","TN","FN"]

    def _wave_key(w):
        m = re.search(r"(\d+)$", str(w))
        return int(m.group(1)) if m else w

    for algo in ["GBM","XGB","RF"]:
        for variant in ["baseline","smote","adasyn","rose"]:
            dfv = res[(res["algo"]==algo) & (res["variant"]==variant)].copy()
            if dfv.empty:
                continue
            dfv = dfv.sort_values("wave", key=lambda s: s.map(_wave_key))
            dfv[cols].to_csv(progress_dir / f"progress_{algo}_{variant}.csv", index=False)

    # Stability report (mean/std/min/max)
    stab = (
        res.groupby(["variant","algo"])[["precision@R","FP_per_1000","pr_auc","roc_auc"]]
          .agg(["mean","std","min","max"]).reset_index()
    )
    # flatten MultiIndex columns
    stab.columns = ["variant","algo"] + ["_".join(col).strip() for col in stab.columns[2:].to_flat_index()]
    stab = stab.sort_values(["algo","variant"])
    stab.to_csv(out_dir / "stability_over_waves.csv", index=False)

    # print
    with pd.option_context("display.max_rows", 200, "display.max_columns", 50):
        print("\n=== SUMMARY (avg over waves) ===")
        display(summary.sort_values(["algo","variant"]))
        print("\n=== PRIMARY KPIs (Precision/Recall focus, sorted) ===")
        display(pr_focus)
        print("\nSaved:", str(out_dir / "results_by_wave.csv"), ",",
              str(out_dir / "summary.csv"), ",",
              str(out_dir / "precision_recall_summary.csv"))
        print("Confusion matrices saved to:", str(conf_dir))
        print("Per-wave progress saved to:", str(progress_dir))
        print("Stability report saved to:", str(out_dir / "stability_over_waves.csv"))

run_comparison()

ROOTS: {'baseline': '4_class_balance/baseline_500', 'smote': '4_class_balance/smote_500', 'adasyn': '4_class_balance/adasyn_500', 'rose': '4_class_balance/rose_500'}


AssertionError: No waves found in baseline. Make sure you generated features/balanced sets first.

In [4]:
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, Tuple, List, Optional

from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, precision_score, recall_score
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

# config
LABEL_COL = "Is_Suspicious"
VARIANT_ROOTS: Dict[str, Path] = {
    "baseline": Path("./4_class_balance/baseline"),
    "smote":    Path("./4_class_balance/smote"),
    "adasyn":   Path("./4_class_balance/adasyn"),
    "rose":     Path("./4_class_balance/rose"),
}
VARIANT_TRAIN_FILE = {
    "baseline": "train_baseline.csv",
    "smote":    "train_smote.csv",
    "adasyn":   "train_adasyn.csv",
    "rose":     "train_rose.csv",
}

VARIANTS_FOR_DIAG = ["baseline", "smote", "adasyn", "rose"]
ALGOS_FOR_DIAG    = ["GBM", "XGB", "RF"]     

TARGET_RECALL = 0.98
RANDOM_STATE  = 42

#ABLATION_DROP_COLS: Optional[List[str]] = None

# helpers
def list_waves(root: Path) -> List[str]:
    if not root.exists():
        return []
    return sorted([p.name for p in root.iterdir() if p.is_dir()])

def load_df(variant: str, wave: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    root = VARIANT_ROOTS[variant]
    d = root / wave
    train_path = _resolve_train_path(d, variant)
    test_path  = d / "test.csv"
    if not train_path.exists() or not test_path.exists():
        raise FileNotFoundError(f"Missing files for {variant}/{wave} → {train_path} / {test_path}")
    tr = pd.read_csv(train_path)
    te = pd.read_csv(test_path)
    if LABEL_COL not in tr.columns or LABEL_COL not in te.columns:
        raise KeyError(f"Column '{LABEL_COL}' not found in train/test for {variant}/{wave}")
    return tr, te


def make_model(algo: str, y_tr: np.ndarray):
    if algo == "GBM":
        return GradientBoostingClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE
        )
    elif algo == "XGB" and HAS_XGB:
        pos = max(1, int((y_tr == 1).sum()))
        neg = max(1, int((y_tr == 0).sum()))
        spw = neg / pos
        return XGBClassifier(
            n_estimators=400, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            reg_lambda=1.0, random_state=RANDOM_STATE,
            eval_metric="logloss", n_jobs=-1, tree_method="hist",
            scale_pos_weight=spw
        )
    elif algo == "RF":
        return RandomForestClassifier(
            n_estimators=300, max_depth=None, min_samples_leaf=1,
            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
        )
    else:
        raise RuntimeError("Unsupported algo or xgboost not installed.")


def threshold_for_recall(y_true: np.ndarray, scores: np.ndarray, target: float):
    order = np.argsort(-scores)
    y_sorted = y_true[order]
    s_sorted = scores[order]
    P = int(y_true.sum())
    if P == 0:
        tn = int((y_true == 0).sum())
        return 1.0, {"precision": 0.0, "recall": 0.0, "tp": 0, "fp": 0, "tn": tn, "fn": 0}
    tp_cum = np.cumsum(y_sorted)
    fp_cum = np.cumsum(1 - y_sorted)
    recall_cum = tp_cum / (P + 1e-12)
    idx = np.where(recall_cum >= target)[0]
    if len(idx) == 0:
        thr = s_sorted[-1] - 1e-12
        pred = np.ones_like(y_true)
    else:
        k = int(idx[0])
        thr = s_sorted[k]
        pred = (scores >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0,1]).ravel()
    prec = tp / max(1, tp + fp)
    rec  = tp / max(1, P)
    return float(thr), {"precision": float(prec), "recall": float(rec), "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn)}

# val-based threshold
def val_threshold_check(variant: str, algo: str, wave: str, val_frac: float = 0.2):
    print(f"\n[VAL CHECK] variant={variant}, algo={algo}, wave={wave}, val_frac={val_frac}")
    tr, te = load_df(variant, wave)

    # Split train chronologically
    n = len(tr)
    n_val = max(1, int(n * val_frac))
    tr_core = tr.iloc[:-n_val].copy()
    val     = tr.iloc[-n_val:].copy()

    print(f"train_core: {len(tr_core)} rows, val: {len(val)} rows, test: {len(te)} rows")

    X_core = tr_core.drop(columns=[LABEL_COL]).to_numpy()
    y_core = tr_core[LABEL_COL].astype(int).to_numpy()
    X_val  = val.drop(columns=[LABEL_COL]).to_numpy()
    y_val  = val[LABEL_COL].astype(int).to_numpy()
    X_test = te.drop(columns=[LABEL_COL]).to_numpy()
    y_test = te[LABEL_COL].astype(int).to_numpy()

    model = make_model(algo, y_core)

    sw = None
    if algo == "GBM":
        n_pos = max(1, int((y_core == 1).sum()))
        n_neg = max(1, int((y_core == 0).sum()))
        w_pos = n_neg / (n_pos + n_neg)
        w_neg = n_pos / (n_pos + n_neg)
        sw = np.where(y_core == 1, w_pos, w_neg)

    model.fit(X_core, y_core, sample_weight=sw) if sw is not None else model.fit(X_core, y_core)

    if hasattr(model, "predict_proba"):
        p_val  = model.predict_proba(X_val)[:, 1]
        p_test = model.predict_proba(X_test)[:, 1]
    else:
        dec_val  = model.decision_function(X_val)
        dec_test = model.decision_function(X_test)
        p_val  = (dec_val  - dec_val.min())  / (dec_val.max()  - dec_val.min()  + 1e-12)
        p_test = (dec_test - dec_test.min()) / (dec_test.max() - dec_test.min() + 1e-12)

    thr_val, at_val = threshold_for_recall(y_val, p_val, TARGET_RECALL)
    pred_test = (p_test >= thr_val).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test, pred_test, labels=[0,1]).ravel()
    prec_test = tp / max(1, tp + fp)
    rec_test  = tp / max(1, int((y_test == 1).sum()))
    fp_per_1000 = fp / max(1, len(y_test)) * 1000.0

    roc_test = roc_auc_score(y_test, p_test)
    prc_test = average_precision_score(y_test, p_test)

    print(f"VAL:  thr={thr_val:.4f}, recall={at_val['recall']:.3f}, precision={at_val['precision']:.3f}")
    print(f"TEST: recall={rec_test:.3f}, precision={prec_test:.3f}, FP={fp}, TP={tp}, TN={tn}, FN={fn}, FP/1000={fp_per_1000:.3f}")
    print(f"      ROC-AUC={roc_test:.4f}, PR-AUC={prc_test:.4f}")

# cross-wave generalisation
def cross_wave_generalisation(variant: str, algo: str, wave_train: str, wave_test: str):
    print(f"\n[CROSS-WAVE] variant={variant}, algo={algo}, train_wave={wave_train}, test_wave={wave_test}")
    tr, _ = load_df(variant, wave_train)
    _, te = load_df(variant, wave_test)

    X_tr = tr.drop(columns=[LABEL_COL]).to_numpy()
    y_tr = tr[LABEL_COL].astype(int).to_numpy()
    X_te = te.drop(columns=[LABEL_COL]).to_numpy()
    y_te = te[LABEL_COL].astype(int).to_numpy()

    model = make_model(algo, y_tr)
    sw = None
    if algo == "GBM":
        n_pos = max(1, int((y_tr == 1).sum()))
        n_neg = max(1, int((y_tr == 0).sum()))
        w_pos = n_neg / (n_pos + n_neg)
        w_neg = n_pos / (n_pos + n_neg)
        sw = np.where(y_tr == 1, w_pos, w_neg)

    model.fit(X_tr, y_tr, sample_weight=sw) if sw is not None else model.fit(X_tr, y_tr)

    if hasattr(model, "predict_proba"):
        p = model.predict_proba(X_te)[:, 1]
    else:
        dec = model.decision_function(X_te)
        p = (dec - dec.min()) / (dec.max() - dec.min() + 1e-12)

    roc = roc_auc_score(y_te, p)
    prc = average_precision_score(y_te, p)
    
    pred05 = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, pred05, labels=[0,1]).ravel()
    prec = tp / max(1, tp + fp)
    rec  = tp / max(1, int((y_te == 1).sum()))

    print(f"AUCs on TEST of {wave_test}: ROC-AUC={roc:.4f}, PR-AUC={prc:.4f}")
    print(f"Threshold=0.5 → recall={rec:.3f}, precision={prec:.3f}, TP={tp}, FP={fp}, TN={tn}, FN={fn}")

# feature ablation
def feature_ablation(variant: str, algo: str, wave: str, drop_cols: Optional[List[str]] = None):
    if not drop_cols:
        print("\n[ABLATION] No columns specified, skipping.")
        return
    print(f"\n[ABLATION] variant={variant}, algo={algo}, wave={wave}, dropping={drop_cols}")
    tr, te = load_df(variant, wave)

    keep_tr = tr.drop(columns=[c for c in drop_cols if c in tr.columns], errors="ignore")
    keep_te = te.drop(columns=[c for c in drop_cols if c in te.columns], errors="ignore")

    X_tr = keep_tr.drop(columns=[LABEL_COL]).to_numpy()
    y_tr = keep_tr[LABEL_COL].astype(int).to_numpy()
    X_te = keep_te.drop(columns=[LABEL_COL]).to_numpy()
    y_te = keep_te[LABEL_COL].astype(int).to_numpy()

    model = make_model(algo, y_tr)
    sw = None
    if algo == "GBM":
        n_pos = max(1, int((y_tr == 1).sum()))
        n_neg = max(1, int((y_tr == 0).sum()))
        w_pos = n_neg / (n_pos + n_neg)
        w_neg = n_pos / (n_pos + n_neg)
        sw = np.where(y_tr == 1, w_pos, w_neg)

    model.fit(X_tr, y_tr, sample_weight=sw) if sw is not None else model.fit(X_tr, y_tr)

    if hasattr(model, "predict_proba"):
        p = model.predict_proba(X_te)[:, 1]
    else:
        dec = model.decision_function(X_te)
        p = (dec - dec.min()) / (dec.max() - dec.min() + 1e-12)

    roc = roc_auc_score(y_te, p)
    prc = average_precision_score(y_te, p)
    thr, at = threshold_for_recall(y_te, p, TARGET_RECALL)

    print(f"After dropping features: ROC-AUC={roc:.4f}, PR-AUC={prc:.4f}")
    print(f"Recall≈{at['recall']:.3f}, Precision≈{at['precision']:.3f} at threshold chosen for Recall≥{TARGET_RECALL}")

# permutation
def permutation_test(variant: str, algo: str, wave: str, n_repeats: int = 3, random_state: int = 42):
    print(f"\n[PERMUTATION TEST] variant={variant}, algo={algo}, wave={wave}, repeats={n_repeats}")
    rng = np.random.default_rng(random_state)
    tr, te = load_df(variant, wave)

    X_tr = tr.drop(columns=[LABEL_COL]).to_numpy()
    y_tr = tr[LABEL_COL].astype(int).to_numpy()
    X_te = te.drop(columns=[LABEL_COL]).to_numpy()
    y_te = te[LABEL_COL].astype(int).to_numpy()

    aucs_roc = []
    aucs_pr  = []

    for i in range(n_repeats):
        y_perm = rng.permutation(y_tr)
        model = make_model(algo, y_perm)

        sw = None
        if algo == "GBM":
            n_pos = max(1, int((y_perm == 1).sum()))
            n_neg = max(1, int((y_perm == 0).sum()))
            w_pos = n_neg / (n_pos + n_neg)
            w_neg = n_pos / (n_pos + n_neg)
            sw = np.where(y_perm == 1, w_pos, w_neg)

        model.fit(X_tr, y_perm, sample_weight=sw) if sw is not None else model.fit(X_tr, y_perm)

        if hasattr(model, "predict_proba"):
            p = model.predict_proba(X_te)[:, 1]
        else:
            dec = model.decision_function(X_te)
            p = (dec - dec.min()) / (dec.max() - dec.min() + 1e-12)

        roc = roc_auc_score(y_te, p)
        prc = average_precision_score(y_te, p)
        aucs_roc.append(roc)
        aucs_pr.append(prc)
        print(f"  repeat {i+1}/{n_repeats}: ROC-AUC={roc:.4f}, PR-AUC={prc:.4f}")

    print(f"Random-label ROC-AUC: mean={np.mean(aucs_roc):.4f}, std={np.std(aucs_roc):.4f}")
    print(f"Random-label PR-AUC : mean={np.mean(aucs_pr):.4f}, std={np.std(aucs_pr):.4f}")

# driver
def run_all_diagnostics():
    for variant in VARIANTS_FOR_DIAG:
        waves = list_waves(VARIANT_ROOTS[variant])
        if not waves:
            print(f"\n[SKIP] No waves found for variant={variant} in", VARIANT_ROOTS[variant])
            continue

        w1 = waves[0]      
        wk = waves[-1]     

        for algo in ALGOS_FOR_DIAG:
            if algo == "XGB" and not HAS_XGB:
                print(f"\n[SKIP] XGB not available for variant={variant}")
                continue

            print("\n" + "="*80)
            print(f"[RUN] Diagnostics for variant={variant}, algo={algo}")
            print("="*80)

            # Threshold
            val_threshold_check(variant, algo, w1, val_frac=0.2)

            # Cross-wave
            if len(waves) > 1:
                cross_wave_generalisation(variant, algo, w1, wk)
                cross_wave_generalisation(variant, algo, wk, w1)

            # Permutation test
            permutation_test(variant, algo, w1, n_repeats=3, random_state=RANDOM_STATE)
run_all_diagnostics()


[RUN] Diagnostics for variant=baseline, algo=GBM

[VAL CHECK] variant=baseline, algo=GBM, wave=synthetic_transactions_structured_100_1, val_frac=0.2
train_core: 6464 rows, val: 1616 rows, test: 2020 rows
VAL:  thr=0.0010, recall=1.000, precision=0.400
TEST: recall=1.000, precision=0.686, FP=16, TP=35, TN=1969, FN=0, FP/1000=7.921
      ROC-AUC=1.0000, PR-AUC=0.9992

[CROSS-WAVE] variant=baseline, algo=GBM, train_wave=synthetic_transactions_structured_100_1, test_wave=synthetic_transactions_structured_100_5
AUCs on TEST of synthetic_transactions_structured_100_5: ROC-AUC=0.9150, PR-AUC=0.6816
Threshold=0.5 → recall=0.467, precision=0.933, TP=14, FP=1, TN=1989, FN=16

[CROSS-WAVE] variant=baseline, algo=GBM, train_wave=synthetic_transactions_structured_100_5, test_wave=synthetic_transactions_structured_100_1
AUCs on TEST of synthetic_transactions_structured_100_1: ROC-AUC=0.9857, PR-AUC=0.6302
Threshold=0.5 → recall=0.286, precision=0.400, TP=10, FP=15, TN=1970, FN=25

[PERMUTATION TEST